In [4]:
# %pip install bs4

In [5]:
# -*- coding: utf-8 -*-
import re
import json
import time
import typing as t
from dataclasses import dataclass, asdict
from pathlib import Path
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup

# =========================
# 데이터 모델
# =========================
@dataclass
class Recipe:
    recipe_id: str
    source_url: str
    title: t.Optional[str]
    ingredients: t.List[str]
    ingredients_struct: t.List[dict]
    steps: t.List[str]
    step_tips: t.List[t.Optional[str]]  # 각 step에 해당하는 tip (없으면 None)
    tips: t.List[str]
    copyright: t.Optional[str]
    # 추가: 요약 정보
    servings: t.Optional[str]
    cook_time: t.Optional[str]
    difficulty: t.Optional[str]

    def to_json(self) -> str:
        return json.dumps(asdict(self), ensure_ascii=False, indent=2)

    def to_ndjson_line(self) -> str:
        return json.dumps(asdict(self), ensure_ascii=False)


# =========================
# 크롤러
# =========================
class RecipeScraper:
    DEFAULT_HEADERS = {
        "User-Agent": "Yorijori-MVP/0.5 (+https://yorijori.example; contact: team@example.com)"
    }
    NOISE_TOKENS = [
        "Steps", "조리순서", "이미지크게보기", "텍스트만보기", "원본보기",
        "맛보장 레시피", "관련 상품", "레시피 작성자", "등록일 :", "#"
    ]

    def __init__(
        self,
        headers: t.Optional[dict] = None,
        base_delay: float = 0.3,
        timeout: int = 15,
        retries: int = 2,
        save_dir: t.Union[str, Path] = ".",
    ) -> None:
        self.session = requests.Session()
        self.session.headers.update(headers or self.DEFAULT_HEADERS)
        self.base_delay = base_delay
        self.timeout = timeout
        self.retries = retries
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)

    # ---------- 유틸 ----------
    @staticmethod
    def _clean_text(s: t.Optional[str]) -> str:
        return " ".join((s or "").split())

    @staticmethod
    def _get_recipe_id_from_url(url: str) -> str:
        m = re.search(r"/recipe/(\d+)", urlparse(url).path)
        return m.group(1) if m else "unknown"

    # ---------- Fetch ----------
    def fetch_html(self, url: str) -> str:
        last_err = None
        for attempt in range(self.retries + 1):
            try:
                resp = self.session.get(url, timeout=self.timeout)
                resp.raise_for_status()
                return resp.text
            except Exception as e:
                last_err = e
                time.sleep(self.base_delay * (2 ** attempt))  # 지수 백오프
        raise last_err

    # ---------- Parse: 단일 페이지 ----------
    def parse(self, url: str, html: str) -> Recipe:
        soup = BeautifulSoup(html, "html.parser")
        rid = self._get_recipe_id_from_url(url)
        title = self._extract_title(soup)
        servings, cook_time, difficulty = self._extract_summary_info(soup)
        ingredients = self._extract_ingredients(soup)
        ingredients_struct = self._extract_ingredients_struct(soup)
        steps, step_tips = self._extract_steps(soup)
        tips = self._extract_tips(soup)
        copyright_owner = self._extract_copyright(soup)

        return Recipe(
            recipe_id=rid,
            source_url=url,
            title=title,
            ingredients=ingredients,
            ingredients_struct=ingredients_struct,
            steps=steps,
            step_tips=step_tips,
            tips=tips,
            copyright=copyright_owner,
            servings=servings,
            cook_time=cook_time,
            difficulty=difficulty,
        )

    # ----- 개별 필드 파서 -----
    def _extract_title(self, soup: BeautifulSoup) -> t.Optional[str]:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            return self._clean_text(og["content"])
        h = soup.select_one("h1, h3")
        return self._clean_text(h.get_text(" ", strip=True)) if h else None

    def _extract_copyright(self, soup: BeautifulSoup) -> t.Optional[str]:
        el = soup.select_one("#contents_area_full > div.view2_pic > div.user_info2 > span")
        return self._clean_text(el.get_text(" ", strip=True)) if el else None

    def _extract_summary_info(self, soup: BeautifulSoup) -> t.Tuple[t.Optional[str], t.Optional[str], t.Optional[str]]:
        """
        #contents_area_full > div.view2_summary.st3 > div.view2_summary_info
        내부 텍스트에서 인분/시간/난이도 문자열을 추출.
        - 보통 3개 토큰(예: '4인분', '60분 이내', '초급')이 순서대로 존재.
        - 레이아웃 변형 대비: 토큰 전체를 스캔해 규칙으로 분류.
        """
        box = soup.select_one("#contents_area_full > div.view2_summary.st3 > div.view2_summary_info")
        if not box:
            return (None, None, None)

        tokens = [self._clean_text(x) for x in box.stripped_strings if self._clean_text(x)]
        servings = cook_time = difficulty = None

        for tok in tokens:
            if servings is None and any(k in tok for k in ["인분", "분량"]):
                servings = tok
                continue
            if cook_time is None and (("분" in tok) or ("시간" in tok) or ("이내" in tok)):
                cook_time = tok
                continue
            if difficulty is None and any(k in tok for k in ["초급", "중급", "고급", "아무나", "쉬움", "보통", "어려움"]):
                difficulty = tok
                continue

        # 혹시 순서대로만 들어있을 경우(토큰 3개) 단순 매핑
        if not any([servings, cook_time, difficulty]) and len(tokens) >= 3:
            servings, cook_time, difficulty = tokens[:3]

        return (servings, cook_time, difficulty)

    def _extract_ingredients(self, soup: BeautifulSoup) -> t.List[str]:
        cont = soup.select_one("#divConfirmedMaterialArea")
        if not cont:
            return []
        items: t.List[str] = []
        for li in cont.select("ul > li"):
            name_a = li.select_one(".ingre_list_name > a")
            qty_span = li.select_one(".ingre_list_ea")
            name = self._clean_text(name_a.get_text(" ", strip=True)) if name_a else None
            qty = self._clean_text(qty_span.get_text(" ", strip=True)) if qty_span else None
            if name == "구매":
                name = None
            if qty and "구매" in qty:
                qty = qty.replace("구매", "").strip() or None
            if name and qty:
                items.append(f"{name} {qty}")
            elif name:
                items.append(name)
            elif qty:
                items.append(qty)
        # 중복 제거
        seen, dedup = set(), []
        for x in items:
            if x and x not in seen:
                seen.add(x)
                dedup.append(x)
        return dedup

    def _extract_ingredients_struct(self, soup: BeautifulSoup) -> t.List[dict]:
        cont = soup.select_one("#divConfirmedMaterialArea")
        if not cont:
            return []
        rows: t.List[dict] = []
        for li in cont.select("ul > li"):
            name_a = li.select_one(".ingre_list_name > a")
            name = self._clean_text(name_a.get_text(" ", strip=True)) if name_a else None
            desc_span = li.select_one(".ingre_list_name > span")
            desc = self._clean_text(desc_span.get_text(" ", strip=True)) if desc_span else None
            qty_span = li.select_one(".ingre_list_ea")
            qty = self._clean_text(qty_span.get_text(" ", strip=True)) if qty_span else None
            if name == "구매":
                name = None
            if qty and "구매" in qty:
                qty = qty.replace("구매", "").strip() or None
            if name or qty:
                rows.append({"name": name, "desc": desc or None, "qty": qty})
        return rows

    def _extract_steps(self, soup: BeautifulSoup) -> t.Tuple[t.List[str], t.List[t.Optional[str]]]:
        """
        조리 단계와 각 단계의 tip을 분리하여 추출.
        Returns: (steps, step_tips) 튜플
        """
        steps: t.List[str] = []
        step_tips: t.List[t.Optional[str]] = []
        
        # .view_step_cont 블록 찾기
        blocks = soup.select(".view_step_cont")
        
        if blocks:
            for block in blocks:
                # .media-body에서 step 텍스트 추출 (step_add 제외)
                media_body = block.select_one(".media-body")
                if media_body:
                    # step_add 요소들을 제거한 후 텍스트 추출
                    media_body_clone = BeautifulSoup(str(media_body), "html.parser")
                    for step_add in media_body_clone.select(".step_add"):
                        step_add.decompose()
                    
                    step_text = self._clean_text(media_body_clone.get_text(" ", strip=True))
                    
                    # step_add에서 tip 추출
                    step_add_elem = block.select_one(".step_add.add_tool")
                    tip_text = None
                    if step_add_elem:
                        # <a> 태그의 텍스트 또는 전체 텍스트
                        tip_link = step_add_elem.select_one("a")
                        if tip_link:
                            tip_text = self._clean_text(tip_link.get_text(" ", strip=True))
                        else:
                            tip_text = self._clean_text(step_add_elem.get_text(" ", strip=True))
                    
                    # step 텍스트가 있는 경우만 추가
                    if step_text:
                        # 노이즈 토큰 체크
                        if any(tok in step_text for tok in self.NOISE_TOKENS):
                            if step_text.strip() in ("Steps", "조리순서"):
                                continue
                        
                        # 숫자 접두사 제거
                        step_text = re.sub(r"^\s*\d+\s*[\.\)]\s*", "", step_text)
                        if step_text:
                            steps.append(step_text)
                            step_tips.append(tip_text if tip_text else None)
        
        # fallback: 기존 로직 (tip 없이 step만 추출)
        if not steps:
            start_node = soup.find(string=lambda s: s and "조리순서" in s)
            if start_node:
                start = start_node.find_parent() or soup
                for sib in start.next_siblings:
                    if not hasattr(sib, "get_text"):
                        continue
                    ttxt = self._clean_text(sib.get_text(" ", strip=True))
                    if not ttxt:
                        continue
                    if any(stop in ttxt for stop in ["팁-주의사항", "관련 상품", "레시피 작성자", "등록일 :"]):
                        break
                    if any(tok in ttxt for tok in self.NOISE_TOKENS):
                        continue
                    ttxt = re.sub(r"^\s*\d+\s*[\.\)]\s*", "", ttxt)
                    if ttxt:
                        steps.append(ttxt)
                        step_tips.append(None)
        
        # 중복 제거 (step과 step_tip을 함께 고려)
        seen_steps = set()
        dedup_steps = []
        dedup_tips = []
        for i, s in enumerate(steps):
            if s and s not in seen_steps:
                seen_steps.add(s)
                dedup_steps.append(s)
                dedup_tips.append(step_tips[i] if i < len(step_tips) else None)
        
        return (dedup_steps, dedup_tips)

    def _extract_tips(self, soup: BeautifulSoup) -> t.List[str]:
        tips: t.List[str] = []
        tip_header = soup.find(string=lambda s: s and "팁-주의사항" in s)
        if tip_header:
            cont = tip_header.find_parent() or soup
            lines = [self._clean_text(x) for x in cont.stripped_strings if self._clean_text(x)]
            try:
                idx = next(i for i, v in enumerate(lines) if "팁-주의사항" in v)
                for ln in lines[idx + 1:]:
                    if "관련 상품" in ln or "등록일 :" in ln:
                        break
                    if any(tok in ln for tok in self.NOISE_TOKENS):
                        continue
                    tips.append(ln)
            except StopIteration:
                pass
        if not tips:
            for dd in soup.select("#obx_recipe_step_start > dl > dd"):
                body = self._clean_text(dd.get_text(" ", strip=True))
                if body and not any(tok in body for tok in self.NOISE_TOKENS):
                    tips.append(body)
        # 중복 제거
        seen, dedup = set(), []
        for tline in tips:
            if tline not in seen:
                seen.add(tline)
                dedup.append(tline)
        return dedup

    # ---------- Save ----------
    def save_recipe_json(self, recipe: Recipe) -> Path:
        out = self.save_dir / f"recipe_{recipe.recipe_id}.json"
        out.write_text(recipe.to_json(), encoding="utf-8")
        return out

    # ---------- Public: 단건 ----------
    def scrape(self, url: str) -> Recipe:
        html = self.fetch_html(url)
        time.sleep(self.base_delay)
        return self.parse(url, html)

    # ---------- Public: 배치 ----------
    def crawl_urls(
        self,
        urls: t.List[str],
        ndjson_path: t.Union[str, Path, None] = None,
        save_each_json: bool = True,
        max_workers: int = 1,
    ) -> t.Dict[str, t.Any]:
        results_ok: t.List[Recipe] = []
        results_fail: t.Dict[str, str] = {}
        ndjson_file = None

        if ndjson_path:
            ndjson_file = Path(ndjson_path)
            ndjson_file.parent.mkdir(parents=True, exist_ok=True)
            if ndjson_file.exists():
                ndjson_file.unlink()

        def _task(u: str):
            try:
                rec = self.scrape(u)
                if save_each_json:
                    self.save_recipe_json(rec)
                if ndjson_file:
                    with ndjson_file.open("a", encoding="utf-8") as f:
                        f.write(rec.to_ndjson_line() + "\n")
                return (u, rec, None)
            except Exception as e:
                return (u, None, str(e))

        if max_workers <= 1:
            for u in urls:
                url, rec, err = _task(u)
                if rec:
                    results_ok.append(rec)
                else:
                    results_fail[url] = err or "unknown error"
        else:
            with ThreadPoolExecutor(max_workers=max_workers) as ex:
                futs = {ex.submit(_task, u): u for u in urls}
                for fut in as_completed(futs):
                    url, rec, err = fut.result()
                    if rec:
                        results_ok.append(rec)
                    else:
                        results_fail[url] = err or "unknown error"

        return {"ok": results_ok, "fail": results_fail}

In [6]:
# 실행
# =========================
import os

# 현재 작업 디렉토리 내에 새 폴더 생성
CURRENT_DIR = Path(os.getcwd()).resolve()
RAW_DATA_DIR = CURRENT_DIR / "extracted_recipes"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"[INFO] 현재 작업 디렉토리: {CURRENT_DIR}")
print(f"[INFO] 저장 경로: {RAW_DATA_DIR}")

if __name__ == "__main__":
    URLS = [
        "https://www.10000recipe.com/recipe/6873683",
        "https://www.10000recipe.com/recipe/6912220",
        "https://www.10000recipe.com/recipe/6883771",
        "https://www.10000recipe.com/recipe/6983886",
        "https://www.10000recipe.com/recipe/7002443",
        "https://www.10000recipe.com/recipe/6867256",
        "https://www.10000recipe.com/recipe/6905196",
        "https://www.10000recipe.com/recipe/6868573",
        "https://www.10000recipe.com/recipe/6873683",
        "https://www.10000recipe.com/recipe/6872350",
        "https://www.10000recipe.com/recipe/1785098",
        "https://www.10000recipe.com/recipe/6879215",
        "https://www.10000recipe.com/recipe/6912220",
        "https://www.10000recipe.com/recipe/7011936",
        "https://www.10000recipe.com/recipe/3568149",
        "https://www.10000recipe.com/recipe/6876544",
        "https://www.10000recipe.com/recipe/6905743",
        "https://www.10000recipe.com/recipe/6893285",
        "https://www.10000recipe.com/recipe/6876357",
        "https://www.10000recipe.com/recipe/7058121",
        "https://www.10000recipe.com/recipe/6858080",
        "https://www.10000recipe.com/recipe/6867091",
        "https://www.10000recipe.com/recipe/7048974",
        "https://www.10000recipe.com/recipe/913370",
        "https://www.10000recipe.com/recipe/6850020",
        "https://www.10000recipe.com/recipe/6870256",
        "https://www.10000recipe.com/recipe/6865491",
        "https://www.10000recipe.com/recipe/6881450",
        "https://www.10000recipe.com/recipe/6887815",
        "https://www.10000recipe.com/recipe/7045100",
        "https://www.10000recipe.com/recipe/6850626",
        "https://www.10000recipe.com/recipe/6884805",
        "https://www.10000recipe.com/recipe/6833097",
        "https://www.10000recipe.com/recipe/6906655",
        "https://www.10000recipe.com/recipe/7038306",
        "https://www.10000recipe.com/recipe/7059616",
        "https://www.10000recipe.com/recipe/7058594",
        "https://www.10000recipe.com/recipe/6851661",
        "https://www.10000recipe.com/recipe/6845428",
        "https://www.10000recipe.com/recipe/6880798",
        "https://www.10000recipe.com/recipe/6891526",
        "https://www.10000recipe.com/recipe/4164229",
        "https://www.10000recipe.com/recipe/6903507",
        "https://www.10000recipe.com/recipe/6872216",
        "https://www.10000recipe.com/recipe/6915139",
        "https://www.10000recipe.com/recipe/6889570",
        "https://www.10000recipe.com/recipe/6864674",
        "https://www.10000recipe.com/recipe/6861312",
        "https://www.10000recipe.com/recipe/6859263",
        "https://www.10000recipe.com/recipe/6894096",
        "https://www.10000recipe.com/recipe/7044262",
        "https://www.10000recipe.com/recipe/6895723",
        "https://www.10000recipe.com/recipe/6840027",
        "https://www.10000recipe.com/recipe/6851272",
        "https://www.10000recipe.com/recipe/6884873",
        "https://www.10000recipe.com/recipe/6858833",
        "https://www.10000recipe.com/recipe/6901938",
        "https://www.10000recipe.com/recipe/6897772",
        "https://www.10000recipe.com/recipe/6838792",
        "https://www.10000recipe.com/recipe/7048296",
        "https://www.10000recipe.com/recipe/6867636",
        "https://www.10000recipe.com/recipe/6869539",
        "https://www.10000recipe.com/recipe/6838943",
        "https://www.10000recipe.com/recipe/6891816",
        "https://www.10000recipe.com/recipe/6993115",
        "https://www.10000recipe.com/recipe/6872492",
        "https://www.10000recipe.com/recipe/6877896",
        "https://www.10000recipe.com/recipe/6871896",
        "https://www.10000recipe.com/recipe/7040324",
        "https://www.10000recipe.com/recipe/7032089",
        "https://www.10000recipe.com/recipe/6879533",
        "https://www.10000recipe.com/recipe/6843373",
        "https://www.10000recipe.com/recipe/6851792",
        "https://www.10000recipe.com/recipe/6882083",
        "https://www.10000recipe.com/recipe/6862107",
        "https://www.10000recipe.com/recipe/6953648",
        "https://www.10000recipe.com/recipe/6913492",
        "https://www.10000recipe.com/recipe/7056751",
        "https://www.10000recipe.com/recipe/7037475",
        "https://www.10000recipe.com/recipe/6883937",
        "https://www.10000recipe.com/recipe/7037582",
        "https://www.10000recipe.com/recipe/6867648",
        "https://www.10000recipe.com/recipe/7050395",
        "https://www.10000recipe.com/recipe/6886360",
        "https://www.10000recipe.com/recipe/6873935",
        "https://www.10000recipe.com/recipe/6887142",
        "https://www.10000recipe.com/recipe/6832126",
        "https://www.10000recipe.com/recipe/6857593",
        "https://www.10000recipe.com/recipe/6884636",
        "https://www.10000recipe.com/recipe/7044402",
        "https://www.10000recipe.com/recipe/6851866",
        "https://www.10000recipe.com/recipe/7052977",
        "https://www.10000recipe.com/recipe/6883771",
        "https://www.10000recipe.com/recipe/6871892",
        "https://www.10000recipe.com/recipe/6838020",
        "https://www.10000recipe.com/recipe/6856975",
        "https://www.10000recipe.com/recipe/7037991",
        "https://www.10000recipe.com/recipe/6911743",
        "https://www.10000recipe.com/recipe/6881688",
        "https://www.10000recipe.com/recipe/7046432",
        "https://www.10000recipe.com/recipe/6906766",
        "https://www.10000recipe.com/recipe/6858140",
        "https://www.10000recipe.com/recipe/6868260",
        "https://www.10000recipe.com/recipe/6881457",
        "https://www.10000recipe.com/recipe/6896908"
    ]
    # 현재 폴더 내 extracted_recipes 폴더에 저장
    scraper = RecipeScraper(save_dir=RAW_DATA_DIR, base_delay=0.35, retries=2)
    ndjson_path = RAW_DATA_DIR / "all_recipes.ndjson"
    report = scraper.crawl_urls(
        urls=URLS,
        ndjson_path=ndjson_path,
        save_each_json=True,
        max_workers=1,
    )
    print(f"완료: {len(report['ok'])}건, 실패: {len(report['fail'])}건")
    if report["fail"]:
        print(json.dumps(report["fail"], ensure_ascii=False, indent=2))

[INFO] 현재 작업 디렉토리: D:\0.Sogang\6\BJS\코드
[INFO] 저장 경로: D:\0.Sogang\6\BJS\코드\extracted_recipes
완료: 105건, 실패: 0건
